In [ ]:
# =============================================================================
# 02_derived_products.ipynb
# Compute PET, SPEI (3/6/12), FVC, forest mask and forest density from the
# aligned stacks produced by 01_preprocessing.ipynb.
#
# Inputs : /content/drive/MyDrive/mkfe_processed/*.nc
# Outputs: /content/drive/MyDrive/mkfe_derived/*.nc
#          PET, SPEI_3, SPEI_6, SPEI_12, FVC, FOREST_MASK, FOREST_DENSITY
#
# Methods:
#   PET          : Hargreaves (Hargreaves & Samani, 1985)
#                  PET = 0.0023 * Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin)
#   SPEI         : Vicente-Serrano et al. (2010), log-logistic fit
#   FVC          : Dimidiate pixel model
#                  FVC = (NDVI - NDVI_soil) / (NDVI_veg - NDVI_soil)
#   Forest mask  : ESA WorldCover 2021, class 10 (tree cover)
#   Forest density: FVC restricted to the forest mask
# =============================================================================

import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from scipy.stats import gamma, norm
from scipy.optimize import curve_fit

# ---------------- Config ----------------
IN_DIR    = Path('/content/drive/MyDrive/mkfe_processed')
OUT_DIR   = Path('/content/drive/MyDrive/mkfe_derived')
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEASONS   = ['JF', 'MAM', 'JJAS', 'OND']
YEARS     = list(range(1995, 2026))
CRS       = 'EPSG:21037'

# Season length in months (for PET and SPEI time scales)
SEASON_MONTHS = {'JF': 2, 'MAM': 3, 'JJAS': 4, 'OND': 3}

# AOI centroid latitude (for extraterrestrial radiation Ra)
LAT_DEG   = -0.15    # Mt. Kenya, slightly south of the equator

# SPEI scales (in seasons — 3/6/12 months → 1/2/4 seasons on our index)
SPEI_SCALES = {'SPEI_3': 1, 'SPEI_6': 2, 'SPEI_12': 4}

In [ ]:
# =============================================================================
# Load the QA-filtered NDVI/LST and all raw stacks from Notebook 1.
# =============================================================================

ndvi      = xr.open_dataarray(IN_DIR / 'NDVI_stack_qa.nc')
lst       = xr.open_dataarray(IN_DIR / 'LST_stack_qa.nc')
sm_l1     = xr.open_dataarray(IN_DIR / 'SM_L1_stack.nc')
sm_l2     = xr.open_dataarray(IN_DIR / 'SM_L2_stack.nc')
sm_l3     = xr.open_dataarray(IN_DIR / 'SM_L3_stack.nc')
sm_l4     = xr.open_dataarray(IN_DIR / 'SM_L4_stack.nc')
t2m       = xr.open_dataarray(IN_DIR / 'T2M_stack.nc')
t2m_max   = xr.open_dataarray(IN_DIR / 'T2M_MAX_stack.nc')
t2m_min   = xr.open_dataarray(IN_DIR / 'T2M_MIN_stack.nc')
precip    = xr.open_dataarray(IN_DIR / 'PRECIP_stack.nc')
srtm      = xr.open_dataarray(IN_DIR / 'SRTM_stack.nc')

print('NDVI  :', ndvi.shape)
print('T2M   :', t2m.shape)
print('PRECIP:', precip.shape)
print('SRTM  :', srtm.shape)

In [ ]:
# =============================================================================
# Compute Ra (MJ m-2 day-1) as a function of day-of-year and latitude.
# FAO-56 formulation.
# =============================================================================

def Ra_mj_m2_day(doy, lat_deg):
    """Extraterrestrial radiation. FAO-56 eq. 21."""
    Gsc = 0.0820
    phi = np.deg2rad(lat_deg)
    dr  = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(delta), -1, 1))
    Ra = (24 * 60 / np.pi) * Gsc * dr * (
        ws * np.sin(phi) * np.sin(delta) +
        np.cos(phi) * np.cos(delta) * np.sin(ws)
    )
    return Ra

# Mid-season day of year for each season
SEASON_DOY = {'JF': 45, 'MAM': 120, 'JJAS': 227, 'OND': 319}

Ra_by_season = {s: Ra_mj_m2_day(SEASON_DOY[s], LAT_DEG) for s in SEASONS}
print('Ra (MJ m-2 d-1):')
for s, v in Ra_by_season.items():
    print(f'  {s:5s}: {v:.2f}')

In [ ]:
# =============================================================================
# PET = 0.0023 * Ra * (Tmean + 17.8) * sqrt(Tmax - Tmin)   [mm/day]
# Then multiply by number of days in the season.
# =============================================================================

SEASON_DAYS = {'JF': 59, 'MAM': 92, 'JJAS': 122, 'OND': 92}

def pet_hargreaves(t_mean, t_max, t_min, ra, n_days):
    """Return seasonal PET in mm. Temperatures in Celsius."""
    dtr = np.clip(t_max - t_min, 0.1, None)   # avoid zero/negative range
    pet_daily = 0.0023 * ra * (t_mean + 17.8) * np.sqrt(dtr)
    pet_daily = np.clip(pet_daily, 0, None)   # PET cannot be negative
    return pet_daily * n_days

# Allocate output
pet = xr.full_like(t2m, np.nan, dtype='float32')
pet.name = 'PET'
pet.attrs.update(units='mm/season', method='Hargreaves')

for i, (year, season) in enumerate(tqdm(t2m.time.to_index(), desc='PET')):
    tm = t2m.isel(time=i).values
    tx = t2m_max.isel(time=i).values
    tn = t2m_min.isel(time=i).values
    pet.values[i] = pet_hargreaves(
        tm, tx, tn,
        Ra_by_season[season],
        SEASON_DAYS[season]
    )

pet.to_netcdf(OUT_DIR / 'PET_stack.nc', engine='netcdf4')
print(f'✓ PET saved. Range: {float(pet.min()):.1f}–{float(pet.max()):.1f} mm/season')

In [ ]:
# =============================================================================
# D (water balance deficit) = precipitation - PET.
# Used as the input to SPEI.
# =============================================================================

# Precip in mm per season (already summed in GEE).
# But CHIRPS gives mm/day... check the export: your GEE CHIRPS block used
# .sum() over daily images, so PRECIP is total mm per season. Good.

D = (precip - pet).rename('D')
D.attrs['units'] = 'mm/season'
D.attrs['note']  = 'Water balance = P - PET'
D.to_netcdf(OUT_DIR / 'D_stack.nc', engine='netcdf4')
print(f'✓ D saved. Range: {float(D.min()):.1f}–{float(D.max()):.1f}')

In [ ]:
# =============================================================================
# SPEI via the log-logistic distribution (Vicente-Serrano et al. 2010).
#
# We work on the seasonal index — each "step" in the time dimension is
# a season. Because seasons have different lengths, we weight each season
# by its month count when computing rolling sums.
# =============================================================================

def loglogistic_cdf(x, alpha, beta, gamma_):
    """CDF of the 3-parameter log-logistic distribution."""
    return 1 / (1 + (alpha / (x - gamma_)) ** beta)

def spei_from_series(series, scale_steps, season_weights):
    """
    series: 1D array of D values (with NaN for missing seasons)
    scale_steps: rolling window length in 'season-steps' (already accounting
                 for seasons being 2/3/4/3 months)
    season_weights: array of month-counts per season (same length as series)
    """
    # Weighted rolling sum: approximate SPEI's month-based accumulation
    # on our season-based index.
    w = season_weights.copy()
    valid = ~np.isnan(series)
    if valid.sum() < 20:
        return np.full_like(series, np.nan, dtype='float32')

    # Rolling sum over scale_steps * season-weight units
    acc = np.full_like(series, np.nan)
    for i in range(len(series)):
        lo = max(0, i - scale_steps + 1)
        window = series[lo:i+1]
        ww     = w[lo:i+1]
        if np.any(~np.isfinite(window)):
            continue
        acc[i] = np.average(window, weights=ww) * ww.sum()

    # Fit log-logistic to the accumulated values
    x = acc[np.isfinite(acc)]
    if len(x) < 20:
        return np.full_like(series, np.nan, dtype='float32')

    # Shift so all values > 0
    shift = -np.min(x) + 0.01
    x_shift = x + shift

    # Method of moments for log-logistic
    try:
        alpha, beta, gamma_ = 0.0, 0.0, 0.0
        # Fit via scipy
        def cdf_fit(xx, a, b, g):
            return loglogistic_cdf(xx, a, b, g)
        p0 = [np.std(np.log(x_shift)), 2.0, 0.0]
        popt, _ = curve_fit(
            cdf_fit, x_shift,
            np.linspace(0.01, 0.99, len(x_shift)),
            p0=p0, maxfev=10000
        )
        alpha, beta, gamma_ = popt
    except Exception:
        return np.full_like(series, np.nan, dtype='float32')

    # Transform to SPEI (standard normal)
    p = loglogistic_cdf(acc + shift, alpha, beta, gamma_)
    p = np.clip(p, 1e-6, 1 - 1e-6)
    spei = norm.ppf(p)
    return spei.astype('float32')

# Get season weights aligned to time index
season_weights = np.array([SEASON_MONTHS[s] for _, s in D.time.to_index()])

# Loop over pixel rows — this is the slow part. For a 3000x4000 grid,
# expect ~1-2 hours on Colab. If too slow, coarsen or subset to AOI core.
ny, nx = D.shape[1], D.shape[2]
D_vals = D.values.astype('float32')

for spei_name, scale in SPEI_SCALES.items():
    print(f'\n▶ Computing {spei_name} (scale={scale} season-steps) ...')
    out = np.full_like(D_vals, np.nan)

    for j in tqdm(range(ny), desc=spei_name):
        for k in range(nx):
            series = D_vals[:, j, k]
            if np.all(np.isnan(series)):
                continue
            out[:, j, k] = spei_from_series(series, scale, season_weights)

    da = xr.DataArray(
        out,
        dims=('time', 'y', 'x'),
        coords={'time': D.time, 'y': D.y, 'x': D.x},
        name=spei_name,
        attrs={'units': 'standardized', 'method': 'log-logistic SPEI'},
    )
    da.rio.write_crs(CRS, inplace=True)
    da.to_netcdf(OUT_DIR / f'{spei_name}_stack.nc', engine='netcdf4')
    print(f'  ✓ {spei_name} saved.')

In [ ]:
# =============================================================================
# FVC = (NDVI - NDVI_soil) / (NDVI_veg - NDVI_soil)
#
# Endmembers per season from NDVI percentiles across the AOI:
#   NDVI_soil = 5th percentile (bare ground)
#   NDVI_veg  = 95th percentile (dense canopy)
# This adapts to seasonal phenology — more correct than fixed values.
# =============================================================================

fvc = xr.full_like(ndvi, np.nan, dtype='float32')
fvc.name = 'FVC'

for i, (year, season) in enumerate(tqdm(ndvi.time.to_index(), desc='FVC')):
    nd = ndvi.isel(time=i).values
    valid = nd[np.isfinite(nd)]
    if len(valid) < 100:
        continue

    ndvi_soil = np.percentile(valid, 5)
    ndvi_veg  = np.percentile(valid, 95)
    denom = ndvi_veg - ndvi_soil
    if denom <= 0:
        continue

    f = (nd - ndvi_soil) / denom
    f = np.clip(f, 0, 1)
    fvc.values[i] = f

fvc.attrs.update(units='0-1', method='dimidiate pixel model')
fvc.to_netcdf(OUT_DIR / 'FVC_stack.nc', engine='netcdf4')
print(f'✓ FVC saved. Range: {float(fvc.min()):.3f}–{float(fvc.max()):.3f}')

In [ ]:
# =============================================================================
# ESA WorldCover 2021 — download the 3x3 degree tile covering Mt. Kenya,
# extract class 10 (tree cover), resample to master grid.
#
# Tile naming for WorldCover v200: N00E036 (equator + 36E).
# =============================================================================

import requests, tarfile, io

# The tile that covers Mt. Kenya (lat -1..0, lon 36..39)
TILE = 'N00E036'
URL = f'https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_{TILE}_Map.tif'

# Simple: read directly via rioxarray from URL
print(f'Downloading WorldCover tile {TILE} ...')
wc = rxr.open_rasterio(URL, masked=True).squeeze()

# Reproject to master grid, nearest neighbour (categorical)
print('Reprojecting WorldCover to master grid ...')
wc_aligned = wc.rio.reproject(
    CRS,
    shape=ndvi.shape[1:],
    transform=ndvi.rio.transform(),
    resampling=2,   # 2 = nearest in rioxarray
)

# Class 10 = tree cover
forest_mask = (wc_aligned == 10).astype('float32')
forest_mask = forest_mask.where(forest_mask == 1)   # 1 inside forest, NaN outside
forest_mask.name = 'FOREST_MASK'
forest_mask.attrs['source'] = 'ESA WorldCover 2021 v200, class 10'
forest_mask.rio.write_crs(CRS, inplace=True)
forest_mask.to_netcdf(OUT_DIR / 'FOREST_MASK.nc', engine='netcdf4')

frac = float((forest_mask == 1).sum()) / forest_mask.size
print(f'✓ Forest mask saved. Forest covers {frac:.1%} of AOI.')

In [ ]:
# =============================================================================
# Forest density = FVC restricted to the forest mask.
# Values near 0 = sparse/degraded forest; values near 1 = dense canopy.
# =============================================================================

forest_density = fvc.where(forest_mask == 1)
forest_density.name = 'FOREST_DENSITY'
forest_density.attrs.update(
    units='0-1',
    note='FVC masked to ESA WorldCover tree-cover class',
)
forest_density.to_netcdf(OUT_DIR / 'FOREST_DENSITY_stack.nc', engine='netcdf4')
print(f'✓ Forest density saved.')

In [ ]:
# =============================================================================
# Summary table + preview of key derived products.
# =============================================================================

summary = []
for f in sorted(OUT_DIR.glob('*.nc')):
    da = xr.open_dataarray(f)
    vals = da.values
    finite = np.isfinite(vals)
    summary.append({
        'file':     f.name,
        'shape':    da.shape,
        'valid_%':  round(100 * finite.sum() / finite.size, 1),
        'min':      float(np.nanmin(vals)) if finite.any() else np.nan,
        'max':      float(np.nanmax(vals)) if finite.any() else np.nan,
    })

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))
summary_df.to_csv(OUT_DIR / 'derived_summary.csv', index=False)

# Preview: PET JF 2010, FVC JJAS 2010, forest density JF 2010
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

pet_sel  = pet.sel(time=('2010', 'JF')).squeeze()
fvc_sel  = fvc.sel(time=('2010', 'JJAS')).squeeze()
fd_sel   = forest_density.sel(time=('2010', 'JF')).squeeze()

axes[0].imshow(pet_sel, cmap='YlOrRd')
axes[0].set_title('PET — JF 2010 (mm/season)')

axes[1].imshow(fvc_sel, cmap='YlGn', vmin=0, vmax=1)
axes[1].set_title('FVC — JJAS 2010')

axes[2].imshow(fd_sel, cmap='Greens', vmin=0, vmax=1)
axes[2].set_title('Forest density — JF 2010')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUT_DIR / 'derived_preview.png', dpi=120)
plt.show()